In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ជំហាន ១៖ កំណត់ម៉ូដែល Pydantic សម្រាប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ

ម៉ូដែលទាំងនេះកំណត់ **ស្កីម៉ា** ដែលភ្នាក់ងារនឹងត្រឡប់មកវិញ។ ការប្រើ `response_format` ជាមួយ Pydantic នាំឲ្យមាន៖
- ✅ ការដកស្រង់ទិន្នន័យដែលមានសុវត្ថិភាពប្រភេទ
- ✅ ការផ្ទៀងផ្ទាត់ស្វ័យប្រវត្តិ
- ✅ គ្មានកំហុសបកប្រែពីចម្លើយអត្ថបទសេរី
- ✅ ការបញ្ជូនមុខងារជាមួយលក្ខខណ្ឌងាយស្រួលដោយផ្អែកលើវាល


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ជំហានទី ២៖ បង្កើតឧបករណ៍ការកក់សណ្ឋាគារ

ឧបករណ៍នេះគឺជាអ្វីដែល **availability_agent** នឹងហៅដើម្បីពិនិត្យមើលថាតើបន្ទប់មានស្រាប់ឬទេ។ យើងប្រើ `@ai_function` decorator ដើម្បី៖
- បម្លែងមុខងារ Python មួយទៅជាឧបករណ៍ដែលអាចហៅតាម AI បាន
- បង្កើតសេចក្តីពិពណ៌នារោងចក្រ JSON ដោយស្វ័យប្រវត្តិសម្រាប់ LLM
- គ្រប់គ្រងការផ្ទៀងផ្ទាត់ប៉ារ៉ាម៉ែត្រ
- អនុញ្ញាតឲ្យ agent ហៅដោយស្វ័យប្រវត្តិ

សម្រាប់ការបង្ហាញនេះ៖
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → មានបន្ទប់ ✅
- **ទីក្រុងផ្សេងទៀតទាំងអស់** → មិនមានបន្ទប់ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ជំហានទី 3: កំណត់មុខងារប៊ិចខណ្ឌសម្រាប់ការបញ្ជាទិស

មុខងារទាំងនេះពិនិត្យមើលចម្លើយរបស់ភ្នាក់ងារ និងកំណត់ថាត្រូវដើរតាមផ្លូវណា ក្នុងលំហទំនើបជំហាន។

**លំនាំសំខាន់៖**
1. ពិនិត្យមើលថា សារ មានប្រភេទជា `AgentExecutorResponse` ឬទេ
2. ព្យាយាមបម្លែងលទ្ធផលដែលមានរចនាសម្ព័ន្ធ (ម៉ូដែល Pydantic)
3. បង្រួមបង្រួច `True` ឬ `False` ដើម្បីគ្រប់គ្រងការបញ្ជាទិស

លំហទំនើបជំហាននឹងវាយតម្លៃលក្ខខណ្ឌទាំងនេះលើ **ខ្សែចំណុច** ដើម្បីសម្រេចចិត្តថាតើត្រូវអនុវត្តអ្នកអនុវត្តណាទៅបន្ទាប់។


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## ជំហាន 4៖ បង្កើតអ្នកអនុវត្តការបង្ហាញផ្ទាល់ខ្លួន

អ្នកអនុវត្តគឺជាផ្នែកធ្វើការ workflow ដែលបំពេញការបម្លែងឬផលប៉ះពាល់ផ្នែកចំហៀង។ យើងប្រើ `@executor` decorator ដើម្បីបង្កើតអ្នកអនុវត្តផ្ទាល់ខ្លួនដែលបង្ហាញលទ្ធផលចុងក្រោយ។

**មូលដ្ឋានគំនិតសំខាន់៖**
- `@executor(id="...")` - ចុះបញ្ជីមុខងារជាអ្នកអនុវត្ត workflow
- `WorkflowContext[Never, str]` - ការពិពណ៌នាប្រភេទសម្រាប់ការបញ្ចូល/ចេញ
- `ctx.yield_output(...)` - បង្ហាញលទ្ធផលចុងក្រោយនៃ workflow


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ជំហានទី 5៖ បន្ទុកអថេរបរិយាកាស

កំណត់រចនាសម្ព័ន្ធភាគីអតិថិជន LLM ។ ឧទាហរណ៍នេះដំណើរការជាមួយ៖
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — ទន្រ្ទាន)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ជំហានទី 6៖ បង្កើតភ្នាក់ងារបច្ចេកវិទ្យាប្រាជ្ញាសិប្បនិម្មិតជាមួយលទ្ធផលរចនាសម្ព័ន្ធ

យើងបង្កើត **ភ្នាក់ងារប្រីសេស្បនិម្មិតបីនាក់**, ដែលរៀបចំក្នុង `AgentExecutor`៖

1. **availability_agent** - ពិនិត្យភាពមានមុខសណ្ឋាគារតាមប្រើឧបករណ៍
2. **alternative_agent** - ស្នើរសុំទីក្រុងជំនួស (ពេលគ្មានបន្ទប់)
3. **booking_agent** - ជំរុញការកក់បន្ទប់ (ពេលមានបន្ទប់)

**លក្ខណៈសម្បត្តិសំខាន់៖**
- `tools=[hotel_booking]` - ផ្ដល់ឧបករណ៍ទៅភ្នាក់ងារ
- `response_format=PydanticModel` - បង្ខំនូវលទ្ធផល JSON រចនាសម្ព័ន្ធ
- `AgentExecutor(..., id="...")` - រៀបចំភ្នាក់ងារសម្រាប់ប្រើប្រាស់ក្នុងលំហូរងារ


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ជំហានទី 7៖ បង្កើតចរន្តការងារជាមួយកម្រិតលក្ខខណ្ឌ

ឥឡូវនេះយើងប្រើ `WorkflowBuilder` ដើម្បីបង្កើតក្រាហ្វដោយមានការបញ្ជូនតាមលក្ខខណ្ឌ៖

**រចនាសម្ព័ន្ធ Workflow៖**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**វិធីសាស្ត្រសំខាន់ៗ៖**
- `.set_start_executor(...)` - កំណត់ចំណុចចូល
- `.add_edge(from, to, condition=...)` - បន្ថែមដែនលក្ខខណ្ឌ
- `.build()` - សម្រេចចរន្តការងារ


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ជំហានទី 8: ប្រតិបត្តិការសាកល្បងទី 1 - ទីក្រុងដោយគ្មានភាពរត់ដោយសារ Availability (Paris)

តោះសាកល្បងផ្លូវ **គ្មានភាពរត់** ដោយស្នើសុំពីសណ្ឋាគារនៅទីក្រុង Paris (ដែលគ្មានបន្ទប់នៅក្នុងការស្ទង់មតិតំណាងរបស់យើង)។  


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ជំហានទី 9: ប្រតិបត្តិការសាកល្បងករណីទី 2 - ទីក្រុង មានភាពអាចប្រើបាន (ស្តុកហុលំ)

ឥឡូវនេះចាំបង្រៀនផ្លូវ **ភាពអាចប្រើបាន** ដោយស្នើសុំសណ្ឋាគារនៅស្តុកហុលំ (ដែលមានបន្ទប់ក្នុងការសមហេតុការណ៍របស់យើង)។


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ចំណុចសំខាន់ និង ជំហានបន្ទាប់

### ✅ អ្វីដែលអ្នកបានរៀន៖

1. **គំរូ WorkflowBuilder**
   - ប្រើ `.set_start_executor()` ដើម្បីកំណត់ចំណុចចូល
   - ប្រើ `.add_edge(from, to, condition=...)` សម្រាប់ផ្លូវលក្ខខណ្ឌ
   - ហៅ `.build()` ដើម្បីបញ្ចប់ workflow

2. **ផ្លូវលក្ខខណ្ឌ**
   - មុខងារលក្ខខណ្ឌពិនិត្យ `AgentExecutorResponse`
   - ពិនិត្យលទ្ធផលដែលមានរចនាសម្ព័ន្ធដើម្បីធ្វើការសម្រេចផ្លូវ
   - បង្វิก `True` ដើម្បីបើកផ្លូវ, `False` ដើម្បីរំលងវា

3. **ការបញ្ចូលឧបករណ៍**
   - ប្រើ `@ai_function` ដើម្បីបម្លែងមុខងារ Python ទៅជាឧបករណ៍ AI
   - Agents ហៅឧបករណ៍ដោយស្វ័យប្រវត្តិពេលត្រូវការ
   - ឧបករណ៍តបដោយ JSON ដែល agents អាចបកប្រែ

4. **លទ្ធផលមានរចនាសម្ព័ន្ធ**
   - ប្រើម៉ូឌែល Pydantic សម្រាប់ការដកទិន្នន័យទាយប្រភេទបានសុវត្ថិភាព
   - បញ្ជាក់ `response_format=MyModel` ពេលបង្កើត agents
   - បកប្រែលទ្ធផលជាមួយ `Model.model_validate_json()`

5. **អ្នកបង្កប់ផ្ទាល់ខ្លួន**
   - ប្រើ `@executor(id="...")` ដើម្បីបង្កើតមុខងារផ្នែក workflow
   - អ្នកបង្កប់អាចបម្លែងទិន្នន័យ ឬអនុវត្តប្រតិបត្តិការផ្លូវក្បាល
   - ប្រើ `ctx.yield_output()` ដើម្បីបង្កើតលទ្ធផល workflow

### 🚀 ការអនុវត្តក្នុងជីវិតពិត៖

- **កក់ដំណើរកំសាន្ត**៖ ពិនិត្យភាពអាចប្រើបាន, ស្នើជម្រើសផ្លាស់ប្តូរ, ប្រៀបធៀបជម្រើស
- **សេវាកម្មអតិថិជន**៖ ផ្លាស់ប្តូរពួនផ្អែកលើប្រភេទបញ្ហា, អារម្មណ៍, អាទិភាព
- **ពាណិជ្ជកម្មអេឡិចត្រូនិច**៖ ពិនិត្យស្តុកទំនិញ, ស្នើជម្រើសផ្លាស់ប្តូរ, ដំណើរការបញ្ជាទិញ
- **ការត្រួតពិនិត្យមាតិការសារព័ត៌មាន**៖ ផ្លាស់ប្តូរពૂતផ្អែកលើពិន្ទុខ្នាត់, ពាក្យប្រើប្រាស់របស់អ្នកប្រើ
- **Workflow អនុម័ត**៖ ផ្លាស់ប្តូរពួនផ្អែកលើចំនួនទឹកប្រាក់, តំណែងអ្នកប្រើ, កម្រិតហានិភ័យ
- **ការ ដំណើរការជាច្រើនដំណាក់កាល**៖ ផ្លាស់ប្តូរពួនផ្អែកលើគុណភាពទិន្នន័យ, សុពលភាព

### 📚 ជំហានបន្ទាប់៖

- បន្ថែមលក្ខខណ្ឌស្មុគស្មាញ (លក្ខខណ្ឌច្រើន)
- អនុវត្តរង្វិលជាមួយការគ្រប់គ្រងស្ថានភាព workflow
- បន្ថែម sub-workflows សម្រាប់ផ្នែកដែលអាចប្រើបានជាមួយគ្នា
- បញ្ចូលជាមួយ API ពិតប្រាកដ (កក់សណ្ឋាគារ, ប្រព័ន្ធស្តុកទំនិញ)
- បន្ថែមការដោះស្រាយកំហុស និងផ្លូវជំនួស
- បង្ហាញ workflow ជាមួយឧបករណ៍ទស្សនាកម្មក្នុងตัว


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
